In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from tqdm import tqdm

print("TF version:", tf.__version__)

# Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize to float32 [0,1]
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Add channel dim for CNN: (N,28,28) -> (N,28,28,1)
x_train = np.expand_dims(x_train, -1)
x_test  = np.expand_dims(x_test, -1)

print(x_train.shape, x_test.shape)

TF version: 2.20.0
(60000, 28, 28, 1) (10000, 28, 28, 1)


In [2]:
def build_cnn():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3,3), activation="relu"),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation="relu"),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy()],
    )
    return model

model = build_cnn()
model.summary()

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("Keras model test acc:", test_acc)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 37ms/step - loss: 0.2232 - sparse_categorical_accuracy: 0.9354 - val_loss: 0.0728 - val_sparse_categorical_accuracy: 0.9790
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - loss: 0.0595 - sparse_categorical_accuracy: 0.9819 - val_loss: 0.0446 - val_sparse_categorical_accuracy: 0.9872
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 40ms/step - loss: 0.0404 - sparse_categorical_accuracy: 0.9872 - val_loss: 0.0395 - val_sparse_categorical_accuracy: 0.9878
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 33ms/step - loss: 0.0311 - sparse_categorical_accuracy: 0.9902 - val_loss: 0.0391 - val_sparse_categorical_accuracy: 0.9897
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 0.0250 - sparse_categorical_accuracy: 0.9921 - val_loss: 0.0356 - val_sparse_categorical_accuracy: 0.9907
Keras model test acc: 0.991599977016449


In [3]:
os.makedirs("artifacts", exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_fp32 = converter.convert()

fp32_path = "artifacts/mnist_cnn_fp32.tflite"
with open(fp32_path, "wb") as f:
    f.write(tflite_fp32)

print("Saved:", fp32_path, "size(bytes)=", os.path.getsize(fp32_path))

INFO:tensorflow:Assets written to: C:\Users\nguye\AppData\Local\Temp\tmpzmfwok7m\assets


INFO:tensorflow:Assets written to: C:\Users\nguye\AppData\Local\Temp\tmpzmfwok7m\assets


Saved artifact at 'C:\Users\nguye\AppData\Local\Temp\tmpzmfwok7m'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1909325682960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1909325686992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046061200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046061584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046060624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046060240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1909325683152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046061008: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved: artifacts/mnist_cnn_fp32.tflite size(bytes)= 904248


In [4]:
def representative_dataset():
    # Lấy 300 mẫu từ train để calibrate (đủ cho demo)
    for i in range(300):
        # TFLite converter expects a list of input arrays
        yield [x_train[i:i+1]]

In [5]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# bật tối ưu
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# calibrate bằng representative dataset
converter.representative_dataset = representative_dataset

# ép input/output int8 (đúng kiểu MCU-friendly)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8 = converter.convert()

int8_path = "artifacts/mnist_cnn_int8.tflite"
with open(int8_path, "wb") as f:
    f.write(tflite_int8)

print("Saved:", int8_path, "size(bytes)=", os.path.getsize(int8_path))

INFO:tensorflow:Assets written to: C:\Users\nguye\AppData\Local\Temp\tmprxj9bqdv\assets


INFO:tensorflow:Assets written to: C:\Users\nguye\AppData\Local\Temp\tmprxj9bqdv\assets


Saved artifact at 'C:\Users\nguye\AppData\Local\Temp\tmprxj9bqdv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1909325682960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1909325686992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046061200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046061584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046060624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046060240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1909325683152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1910046061008: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\ml\kaggle-ml-basics\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved: artifacts/mnist_cnn_int8.tflite size(bytes)= 235848


In [6]:
def tflite_evaluate(tflite_model_path, x_data, y_data, max_samples=2000):
    # Load interpreter
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index = input_details[0]["index"]
    output_index = output_details[0]["index"]

    # Quantization params (nếu int8)
    in_dtype = input_details[0]["dtype"]
    out_dtype = output_details[0]["dtype"]

    in_scale, in_zero = input_details[0].get("quantization", (1.0, 0))
    out_scale, out_zero = output_details[0].get("quantization", (1.0, 0))

    correct = 0
    n = min(max_samples, len(x_data))

    for i in tqdm(range(n)):
        x = x_data[i:i+1]

        # Nếu model int8: cần quantize input từ float32 sang int8
        if in_dtype == np.int8:
            # x đang float32 [0,1]
            x_q = np.clip(np.round(x / in_scale + in_zero), -128, 127).astype(np.int8)
            interpreter.set_tensor(input_index, x_q)
        else:
            interpreter.set_tensor(input_index, x.astype(in_dtype))

        interpreter.invoke()
        out = interpreter.get_tensor(output_index)

        # Nếu output int8: dequantize để dễ hiểu (không bắt buộc để argmax)
        # Argmax có thể làm trực tiếp trên int8 cũng được.
        pred = int(np.argmax(out, axis=1)[0])

        if pred == int(y_data[i]):
            correct += 1

    return correct / n, {
        "input_dtype": str(in_dtype),
        "output_dtype": str(out_dtype),
        "in_quant": (in_scale, int(in_zero)),
        "out_quant": (out_scale, int(out_zero)),
        "samples": n
    }

In [9]:
acc_fp32, info_fp32 = tflite_evaluate(fp32_path, x_test, y_test, max_samples=2000)
print("TFLite FP32 acc:", acc_fp32)
print("Info:", info_fp32)

c:\ml\kaggle-ml-basics\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
100%|██████████| 2000/2000 [00:00<00:00, 5408.33it/s]

TFLite FP32 acc: 0.986
Info: {'input_dtype': "<class 'numpy.float32'>", 'output_dtype': "<class 'numpy.float32'>", 'in_quant': (0.0, 0), 'out_quant': (0.0, 0), 'samples': 2000}


In [10]:
acc_int8, info_int8 = tflite_evaluate(int8_path, x_test, y_test, max_samples=2000)
print("TFLite INT8 acc:", acc_int8)
print("Info:", info_int8)

100%|██████████| 2000/2000 [00:00<00:00, 13817.39it/s]

TFLite INT8 acc: 0.985
Info: {'input_dtype': "<class 'numpy.int8'>", 'output_dtype': "<class 'numpy.int8'>", 'in_quant': (0.003921568859368563, -128), 'out_quant': (0.00390625, -128), 'samples': 2000}


In [11]:
size_fp32 = os.path.getsize(fp32_path)
size_int8 = os.path.getsize(int8_path)

print("FP32 size (KB):", size_fp32 / 1024)
print("INT8 size (KB):", size_int8 / 1024)
print("Compression ratio:", size_fp32 / size_int8)

FP32 size (KB): 883.0546875
INT8 size (KB): 230.3203125
Compression ratio: 3.83402869644856


In [12]:
img = x_test[0] * 255  # nếu x_test đã normalize [0,1]
img = img.astype(np.uint8)

print("Label:", y_test[0])
print(img.shape)  # (28,28)

Label: 7
(28, 28, 1)


In [13]:
flat = img.flatten()

with open("mnist_sample0.c", "w") as f:
    f.write("const unsigned char mnist_sample0[784] = {\n")
    for i, v in enumerate(flat):
        f.write(f"{int(v)},")
        if (i+1) % 28 == 0:
            f.write("\n")
    f.write("};\n")

In [14]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_test = x_test.astype("float32") / 255.0

K = 100  # số ảnh bạn muốn nhúng
scale = 0.003921568859368563
zero = -128

# Quantize -> int8
xq = np.round(x_test[:K] / scale + zero).astype(np.int8)   # (K, 28, 28)

# Flatten thành (K, 784)
xq_flat = xq.reshape(K, -1)
y = y_test[:K].astype(np.uint8)

# Xuất ra C
def to_c_array(name, arr, dtype="int8_t", per_line=16):
    flat = arr.flatten()
    s = f"const {dtype} {name}[{len(flat)}] = {{\n  "
    for i, v in enumerate(flat):
        s += str(int(v)) + ", "
        if (i+1) % per_line == 0:
            s += "\n  "
    s += "\n};\n"
    return s

c_images = to_c_array("g_mnist_images_int8", xq_flat, "int8_t")
c_labels = to_c_array("g_mnist_labels", y, "uint8_t")

with open("mnist_test_images.c", "w", encoding="utf-8") as f:
    f.write('#include <stdint.h>\n')
    f.write(f'const int g_mnist_num_images = {K};\n')
    f.write(c_images)
    f.write(c_labels)

with open("mnist_test_images.h", "w", encoding="utf-8") as f:
    f.write('#pragma once\n#include <stdint.h>\n\n')
    f.write('extern const int g_mnist_num_images;\n')
    f.write('extern const int8_t g_mnist_images_int8[]; // size = K*784\n')
    f.write('extern const uint8_t g_mnist_labels[];     // size = K\n')

In [1]:
print(y_test[18])
print(np.argmax(model.predict(x_test[18:19]), axis=1))

NameError: name 'y_test' is not defined